# SmolLM2 Memory Fusion r48/r64 — Evaluation Only

This notebook **does not train anything**.

It reloads the already-trained SmolLM2-135M Memory Fusion checkpoints (r48 and r64), verifies that all 30 Transformer self-attention layers were replaced, and compares them against the original SmolLM2-135M on the same held-out WikiText-2 blocks.

Checkpoint discovery is automatic:
1. Hugging Face model repos (default `vtava/SmolLM2-135M-MemoryFusion-r48` and `...-r64`)
2. Google Drive backup folder, if available
3. manual checkpoint ZIP upload as a fallback

The WikiText loader uses the canonical `Salesforce/wikitext` repository and includes a direct-Parquet fallback for newer Hugging Face URI parsing.


In [ ]:
import os, sys, json, math, gc, pathlib, subprocess, importlib, shutil, zipfile
import torch

subprocess.run(['nvidia-smi'], check=False)

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-e', str(REPO_DIR),
    'transformers==4.57.6',
    'datasets>=3,<5',
    'huggingface_hub>=0.34,<2',
    'pandas',
], check=True)

SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()

from huggingface_hub import snapshot_download, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd

from tinycenn_lm.smollm2_memory_fusion import (
    build_smollm2_memory_fusion,
    structural_summary,
)

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())


## Settings

If your Hugging Face username/repository names are different, change only `HF_MODEL_REPOS`.

If the models were not published but were backed up to Google Drive, set `DRIVE_CHECKPOINT_ROOT` to the folder containing `r48/` and `r64/`.


In [ ]:
BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'
MEMORY_RANKS = [48, 64]

HF_MODEL_REPOS = {
    48: 'vtava/SmolLM2-135M-MemoryFusion-r48',
    64: 'vtava/SmolLM2-135M-MemoryFusion-r64',
}

DRIVE_CHECKPOINT_ROOT = '/content/drive/MyDrive/TinyCeNN-LM/checkpoints/smollm2-memory-fusion'
LOCAL_ROOT = pathlib.Path('/content/memory_fusion_eval_checkpoints')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
EVAL_CONTEXTS = [128, 256]
EVAL_BLOCKS = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = (torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type == 'cuda' else torch.float32))
print('device:', device, 'dtype:', dtype)


## Locate/download r48 and r64 checkpoints

This cell first tries Hugging Face. If those repos are unavailable, it mounts Google Drive and looks for backups there. If neither exists, it asks you to upload checkpoint ZIPs manually.


In [ ]:
REQUIRED = ['smollm2_memory_fusion.pt', 'smollm2_memory_fusion_config.json']

def checkpoint_ok(path):
    path = pathlib.Path(path)
    return all((path / name).exists() for name in REQUIRED)

def copy_checkpoint_dir(src, dst):
    src, dst = pathlib.Path(src), pathlib.Path(dst)
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(src, dst)
    if not checkpoint_ok(dst): raise RuntimeError(f'Incomplete checkpoint copied from {src}')

checkpoint_dirs = {}
for rank in MEMORY_RANKS:
    target = LOCAL_ROOT / f'r{rank}'
    repo_id = HF_MODEL_REPOS[rank]
    try:
        print(f'Trying Hugging Face: {repo_id}')
        downloaded = pathlib.Path(snapshot_download(repo_id=repo_id, repo_type='model', allow_patterns=['smollm2_memory_fusion.pt','smollm2_memory_fusion_config.json','smollm2_memory_fusion_training_report.json','README.md']))
        if checkpoint_ok(downloaded):
            copy_checkpoint_dir(downloaded, target)
            checkpoint_dirs[rank] = target
            print(f'✅ r{rank} loaded from Hugging Face')
    except Exception as exc:
        print(f'HF r{rank} unavailable: {type(exc).__name__}: {exc}')

missing = [r for r in MEMORY_RANKS if r not in checkpoint_dirs]
if missing:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        drive_root = pathlib.Path(DRIVE_CHECKPOINT_ROOT)
        for rank in missing:
            for src in [drive_root / f'r{rank}', drive_root / f'SmolLM2-135M-MemoryFusion-r{rank}']:
                if checkpoint_ok(src):
                    target = LOCAL_ROOT / f'r{rank}'
                    copy_checkpoint_dir(src, target)
                    checkpoint_dirs[rank] = target
                    print(f'✅ r{rank} loaded from Drive: {src}')
                    break
    except Exception as exc:
        print('Drive fallback unavailable:', type(exc).__name__, exc)

missing = [r for r in MEMORY_RANKS if r not in checkpoint_dirs]
if missing:
    print('Missing ranks:', missing, '- upload checkpoint ZIP files now.')
    from google.colab import files
    uploaded = files.upload()
    for filename, blob in uploaded.items():
        zip_path = pathlib.Path('/content') / filename
        zip_path.write_bytes(blob)
        if not zipfile.is_zipfile(zip_path): continue
        extract_root = pathlib.Path('/content') / (zip_path.stem + '_extract')
        if extract_root.exists(): shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(extract_root)
        for rank in list(missing):
            for p in [extract_root, *[x for x in extract_root.rglob('*') if x.is_dir()]]:
                if checkpoint_ok(p):
                    cfg = json.loads((p / 'smollm2_memory_fusion_config.json').read_text())
                    if int(cfg.get('memory_fusion', {}).get('memory_rank', -1)) == rank:
                        target = LOCAL_ROOT / f'r{rank}'
                        copy_checkpoint_dir(p, target)
                        checkpoint_dirs[rank] = target
                        print(f'✅ r{rank} loaded from uploaded ZIP')
                        break

missing = [r for r in MEMORY_RANKS if r not in checkpoint_dirs]
if missing:
    raise RuntimeError('No persistent checkpoint found for ranks ' + ', '.join(map(str, missing)) + '. If the old Colab VM disappeared before publishing/downloading/backing up, those trained weights are gone and must be retrained.')

print(checkpoint_dirs)


## Verify architecture and load WikiText-2 correctly


In [ ]:
for rank in MEMORY_RANKS:
    model = build_smollm2_memory_fusion(checkpoint_dirs[rank], device=device, dtype=dtype)
    summary = structural_summary(model)
    print(f'r{rank}:', summary)
    assert summary['memory_fusion_layers'] == 30
    assert summary['transformer_attention_layers'] == 0
    assert all(layer.self_attn.core.memory_rank == rank for layer in model.model.layers)
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
WIKITEXT_REPO = 'Salesforce/wikitext'
WIKITEXT_CONFIG = 'wikitext-2-raw-v1'
try:
    wiki = load_dataset(WIKITEXT_REPO, WIKITEXT_CONFIG, split='test')
    print('✅ WikiText loaded normally:', len(wiki))
except Exception as exc:
    print('Normal loader failed, using direct Parquet fallback:', type(exc).__name__, exc)
    parquet_path = hf_hub_download(repo_id=WIKITEXT_REPO, repo_type='dataset', filename=f'{WIKITEXT_CONFIG}/test-00000-of-00001.parquet')
    wiki = load_dataset('parquet', data_files={'test': parquet_path}, split='test')
    print('✅ WikiText loaded via Parquet:', len(wiki))

stream = '\n'.join(str(x) for x in wiki['text'] if str(x).strip())
token_ids = tokenizer(stream, add_special_tokens=False)['input_ids']
print('evaluation tokens:', len(token_ids))


## Evaluate original Transformer vs Memory Fusion r48/r64

Lower NLL/perplexity is better. `ΔNLL < 0` means Memory Fusion beats the original SmolLM2 Transformer on this evaluation.


In [ ]:
def make_eval_blocks(context):
    blocks = []
    for start in range(0, len(token_ids) - context + 1, context):
        blocks.append(torch.tensor(token_ids[start:start+context], dtype=torch.long))
        if len(blocks) >= EVAL_BLOCKS: break
    return blocks

@torch.no_grad()
def evaluate_model(model, blocks):
    model.eval(); losses = []
    for block in blocks:
        x = block.unsqueeze(0).to(device)
        out = model(input_ids=x, labels=x, use_cache=False, return_dict=True)
        losses.append(float(out.loss.detach().float()))
    nll = sum(losses) / len(losses)
    return {'nll': nll, 'ppl': math.exp(nll), 'blocks': len(losses)}

eval_rows = []
for context in EVAL_CONTEXTS:
    print('\n' + '=' * 90, '\nCONTEXT', context)
    blocks = make_eval_blocks(context)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype).to(device)
    base.config.use_cache = False
    result = evaluate_model(base, blocks)
    eval_rows.append({'model':'Transformer original','rank':0,'context':context,**result})
    print(f"Transformer        NLL={result['nll']:.6f} PPL={result['ppl']:.4f}")
    del base; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    for rank in MEMORY_RANKS:
        student = build_smollm2_memory_fusion(checkpoint_dirs[rank], device=device, dtype=dtype)
        result = evaluate_model(student, blocks)
        eval_rows.append({'model':f'Memory Fusion r{rank}','rank':rank,'context':context,**result})
        print(f"Memory Fusion r{rank} NLL={result['nll']:.6f} PPL={result['ppl']:.4f}")
        del student; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

eval_df = pd.DataFrame(eval_rows)
for context in EVAL_CONTEXTS:
    base_nll = float(eval_df[(eval_df.context == context) & (eval_df['rank'] == 0)].iloc[0].nll)
    mask = eval_df.context == context
    eval_df.loc[mask, 'delta_nll_vs_transformer'] = eval_df.loc[mask, 'nll'] - base_nll
eval_df = eval_df.sort_values(['context','nll']).reset_index(drop=True)
display(eval_df)

print('\nFINAL RESULT')
for context in EVAL_CONTEXTS:
    print(f'\nContext {context}')
    for _, row in eval_df[eval_df.context == context].sort_values('nll').iterrows():
        delta = float(row['delta_nll_vs_transformer'])
        mark = '🏆' if row['rank'] != 0 and delta < 0 else ''
        print(f"{row['model']:28s} NLL={row['nll']:.6f} PPL={row['ppl']:.4f} ΔNLL={delta:+.6f} {mark}")


## Save/download results


In [ ]:
RESULT_DIR = pathlib.Path('/content/memory_fusion_evaluation_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = RESULT_DIR / 'smollm2_memory_fusion_r48_r64_eval.csv'
json_path = RESULT_DIR / 'smollm2_memory_fusion_r48_r64_eval.json'
eval_df.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(eval_df.to_dict(orient='records'), indent=2), encoding='utf-8')
print(csv_path); print(json_path)
from google.colab import files
files.download(str(csv_path)); files.download(str(json_path))
